> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 10 · Notebook 01 — A baseline first, then better bars and events

**Sessions:** S1–S2 (Framing financial ML · Data structures & event sampling) · [Lesson plan](../../docs/lessons/PART_10_MACHINE_LEARNING.md) · graded labs in [`labs/part10/`](../../labs/part10/)

**You will:**
1. Build the simplest ML dataset (lagged returns → tomorrow's direction) and a walk-forward baseline.
2. See why a model must beat the base rate, not 50%.
3. Build dollar bars and compare their statistics with time bars.
4. Sample events with a CUSUM filter instead of using every day.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure (a hidden regime, a known autocorrelation, planted importances, pure noise), so you can see what a model can learn, and that it learns nothing from noise.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p10lib.py is in notebooks/part10/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p10lib as p

p.use_course_style()

## 1. The baseline

Before any fancy model, set the bar: the last 5 log returns as features, tomorrow's direction as the label. Features at `t` are `lag0 = r_t, lag1 = r_{t−1}, …` (`r.shift(k)`); the target is 1 if `r_{t+1} > 0`; also keep `fwd = r_{t+1}` for the P&L. Drop rows with any NaN (the start, and the last row which has no tomorrow).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def direction_dataset(close, n_lags=5):
    r = np.log(close).diff()
    X = pd.DataFrame({f"lag{k}": r.shift(k) for k in range(n_lags)})
    fwd = r.shift(-1)
    ok = X.notna().all(axis=1) & fwd.notna()
    return X[ok], (fwd[ok] > 0).astype(int), fwd[ok]

rw = p.random_walk_prices(2500)                         # pure noise
mk = p.regime_market()                                  # a hidden trend/chop regime
mine = [p.attempt(direction_dataset, rw), p.attempt(direction_dataset, mk.close)]
mine = p.check("direction_dataset", mine, [p.direction_dataset(rw), p.direction_dataset(mk.close)])
X, y, fwd = mine[0]
X.head(3)

Now a walk-forward logistic regression: every 21 days, refit on everything before and trade the next 21 days (+1 if `p(up) > 0.5`, else −1), paying 5 bp per unit of position change (`p.walk_forward_logit`).

In [ ]:
rows = {}
for name, close in (("random walk", rw), ("regime market", mk["close"])):
    wf = p.walk_forward_logit(*p.direction_dataset(close))
    rows[name] = {"accuracy": wf["accuracy"], "share of up days": wf["always_up"], "net Sharpe": p.sharpe(wf["returns"])}
pd.DataFrame(rows).T.round(3)

On the random walk, the accuracy is exactly one minus the share of up days: the model learned to say "down" every day, the base rate of its training data. Its positive Sharpe is the luck of a random walk that happened to fall. On the regime market, the accuracy is *below* "always up". A 51% accuracy means nothing until it beats the base rate, a simple rule, and costs, out of sample. That is the rule for all of Part 10.

## 2. Bars that follow activity

Time bars sample a quiet lunch hour as often as a busy open, so their returns mix small and large variances: fat tails. **Dollar bars** close each time another fixed amount of money has traded: bar id = cumulative `price × size` `//` threshold. Group the ticks by bar id; each bar is labeled with the time of its **last** trade (a bar is known only when it closes).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def dollar_bars(ticks, threshold):
    dv = (ticks["price"] * ticks["size"]).cumsum()
    bar_id = (dv // threshold).astype(int).to_numpy()
    g = ticks.groupby(bar_id)
    bars = g["price"].agg(open="first", high="max", low="min", close="last")
    bars["volume"] = g["size"].sum()
    bars.index = pd.DatetimeIndex(ticks.index.to_series().groupby(bar_id).last().to_numpy())
    return bars

ticks = p.tick_data()                                   # 30 days of trades with quiet and busy days
tbars = p.time_bars(ticks, "30min")
threshold = (ticks["price"] * ticks["size"]).sum() / len(tbars)      # the same number of bars
dbars = p.check("dollar_bars", p.attempt(dollar_bars, ticks, threshold), p.dollar_bars(ticks, threshold))
pd.DataFrame({"time bars (30 min)": p.bar_stats(tbars["close"]), "dollar bars": p.bar_stats(dbars["close"])}).T.round(3)

In [ ]:
per_day = pd.DataFrame({"time bars": tbars.groupby(tbars.index.date).size(),
                        "dollar bars": dbars.groupby(dbars.index.date).size()})
per_day.plot(kind="bar", figsize=(11, 3), width=0.8, title="bars per day: a fixed clock vs the market's activity")
plt.xticks([]); plt.show()

Same number of bars, very different samples: the dollar bars' returns are much closer to normal (a Jarque–Bera statistic near 1 instead of 16), because each bar carries about the same amount of trading. Busy days get more bars, quiet days fewer.

## 3. Sample events, not days

Most days carry no new information. The symmetric **CUSUM filter** accumulates log returns `x` in two running sums, `s+ = max(0, s+ + x)` and `s− = min(0, s− + x)`, and records an event (and resets that sum) when `s+ > h` or `s− < −h`. Labels are then built only at events.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def cusum_events(close, h):
    events, s_pos, s_neg = [], 0.0, 0.0
    for ts, x in np.log(close).diff().dropna().items():
        s_pos, s_neg = max(0.0, s_pos + x), min(0.0, s_neg + x)
        if s_pos > h:
            s_pos = 0.0
            events.append(ts)
        elif s_neg < -h:
            s_neg = 0.0
            events.append(ts)
    return pd.DatetimeIndex(events)

cases = [(mk.close, 0.015), (mk.close, 0.04)]
mine = [p.attempt(cusum_events, *c) for c in cases]
mine = p.check("cusum_events", mine, [p.cusum_events(*c) for c in cases])
print(f"{len(mk)} days → {len(mine[0])} events with h = 1.5%, {len(mine[1])} with h = 4%")
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(mk["close"].iloc[:500], lw=1)
ev = mine[1][mine[1] <= mk.index[499]]
ax.scatter(ev, mk["close"].loc[ev], color="#eb6834", s=18, zorder=3, label="CUSUM event (h = 4%)")
ax.legend(); ax.set_title("events fire when the price has moved enough, not every day"); plt.show()

## Wrap-up

* Baseline first: every model must beat the base rate and a simple rule, after costs, out of sample.
* Dollar (or volume) bars sample by activity and give better-behaved returns.
* CUSUM events keep the informative moments; fewer, less overlapping labels.
* Graded version: `labs/part10/week33_features` (`walk_forward_logit`, `dollar_bars`, `cusum_events`).